In [27]:
from pathlib import Path
PROJECT_ROOT = Path.cwd().parent

import pandas as pd

pd.set_option('display.max_columns', None)  # none은 제한값 자리(몇개 열까지 보여줄까)
pd.set_option('display.max_rows', None)

# ============================================================
# 0. 데이터 불러오기: 2-5 train_test_comparison에서 제거된 이상치까지 반영된 데이터셋
# ============================================================

train_full = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_train_full_clean.parquet')
test = pd.read_parquet(PROJECT_ROOT / 'parqueted_data' / '0_test_future.parquet')
test_pred = pd.read_csv(PROJECT_ROOT.parent / 'data' / 'result_v1.csv')

from sklearn import set_config
set_config(display='text')
# scikit-learn의 HTML 출력 과정에서 발생한 cp949 간 UTF-8 충돌 때문에 넣는 코드

In [2]:
print(test.shape, test_pred.shape)

(114111, 34) (186263, 2)


In [4]:
test_pred.head(5)
test_pred.info()

<class 'pandas.DataFrame'>
RangeIndex: 186263 entries, 0 to 186262
Data columns (total 2 columns):
 #   Column           Non-Null Count   Dtype  
---  ------           --------------   -----  
 0   row_id           186263 non-null  str    
 1   pred_maxrise_6h  186263 non-null  float64
dtypes: float64(1), str(1)
memory usage: 5.0 MB


In [29]:
# reindex(): 부모 인덱스 기준, 자식을 재정렬하여 가져옴
# 여러 컬럼을 한번에 지정해서 index에 맞게 다 들고 올 수 있음(map이랑 다른부분)
# reindex 용법: 부모[컬럼] = 자식[컬럼].reindex(부모인덱스)
# 부모, 자식 크기가 안맞을때 부모인덱스에 해당하는만큼만 자식에서 가져온다.
# 자식도 같은 인덱스가 지정 돼 있어야 한다.(set_index()를 써서라도)
test['test_pred_v1'] = (
    test_pred
    .set_index('row_id')['pred_maxrise_6h']  # 인덱스로 설정해야 매칭 가능
    # set_index로 인덱스 설정 -> pred_maxrise_6h 불러오기
    .reindex(test.index)
)

# 원본 예측값에 결측치가 없는지 확인
assert not test_pred['pred_maxrise_6h'].isnull().any(), '원본 예측값에 결측치 있음'

# reindex 후 미매칭 여부 확인
assert not test['test_pred_v1'].isnull().any(), '인덱스 미매칭 발생'


# map(): 시리즈나 딕셔너리, 함수로 매핑규칙을 지정, 부모와 자식의 인덱스가 같기만 하면
# 시리즈만 받으므로 여러 열을 가져오지 못한다.
# 매핑규칙으로 부모와 같은 인덱스와 값을 가진 자녀 객체가 있으면 됨

# map 용법: 부모[컬럼] = 부모 인덱스.map(매핑규칙(자녀의 것으로서))  # 부모 인덱스를 매핑규칙에 따라 자식 인덱스와 매칭
# 자식과 부모의 인덱스가 같은 것만 매핑 가능(set_index()를 써서라도)
# test['test_pred_v1'] = (
# test.index  # test는 'row_id'가 이미 인덱스라서 이렇게 들고옴
# .map(test_pred.set_index('row_id')['pred_maxrise_6h']) 
# )

In [7]:
print(test.shape, test_pred.shape)
test.head()

(114111, 35) (186263, 2)


,timestamp,station_id,wl_t,wl_t_minus_1h,wl_t_minus_3h,wl_t_minus_6h,wl_t_minus_12h,wl_t_minus_24h,imerg_rain_1h,imerg_rain_6h,...,wl_diff_3h,wl_diff_6h,wl_diff_12h,wl_diff_24h,imerg_int_1to6h,imerg_int_6to24h,sfc_int_1to6h,sfc_int_6to24h,target_maxrise_6h,test_pred_v1
row_id,,,,,,,,,,,,,,,,,,,,,
TE_000046543,2100-03-12,ST0001,0.19,0.20,0.22,0.25,0.16,0.18,2.333,2.340,...,-0.03,-0.06,0.03,0.01,0.0014,0.0,0.0,0.0,-0.01,0.099438
TE_000111013,2100-03-12,ST0002,0.85,0.84,0.86,0.85,0.84,0.85,0.140,0.140,...,-0.01,0.00,0.01,0.00,0.0000,0.0,0.0,0.0,0.01,0.008376
TE_000086280,2100-03-12,ST0003,0.76,0.76,0.77,0.78,0.77,0.76,2.772,2.773,...,-0.01,-0.02,-0.01,0.00,0.0002,0.0,0.0,0.0,0.01,0.042920
TE_000136298,2100-03-12,ST0005,0.98,0.96,0.96,0.93,0.95,0.97,0.205,0.208,...,0.02,0.05,0.03,0.01,0.0006,0.0,0.0,0.0,0.02,0.040083
TE_000166154,2100-03-12,ST0006,0.73,0.72,0.62,0.64,0.65,0.65,0.655,0.695,...,0.11,0.09,0.08,0.08,0.0080,0.0,0.0,0.0,0.02,0.047127


In [30]:
from sklearn.metrics import root_mean_squared_error

rmse = root_mean_squared_error(
    test['target_maxrise_6h'],
    test['test_pred_v1']
)

print(f'Test RMSE: {rmse:.6f}')

Test RMSE: 0.202310
